In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("test")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/23 21:28:31 WARN Utils: Your hostname, CrisBook.local, resolves to a loopback address: 127.0.0.1; using 192.168.13.159 instead (on interface en0)
26/04/23 21:28:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 21:28:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/23 21:28:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/23 21:28:32 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
import os
import requests

def download_file(url: str, output_path: str):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

In [3]:
import gzip
import shutil

def gunzip_file(input_path: str, output_path: str):
    with gzip.open(input_path, "rb") as f_in:
        with open(output_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

In [4]:
from datetime import datetime

date_str = datetime.now().strftime("%Y-%m-%d")

epss_url = f"https://epss.empiricalsecurity.com/epss_scores-{date_str}.csv.gz"
epss_gz_path = f"../data/raw/epss/epss_scores-{date_str}.csv.gz"
epss_csv_path = f"../data/raw/epss/epss_scores-{date_str}.csv"

download_file(epss_url, epss_gz_path)
gunzip_file(epss_gz_path, epss_csv_path)

print("EPSS downloaded and extracted successfully")

EPSS downloaded and extracted successfully


In [5]:
epss_raw = (
    spark.read
    .option("header", True)
    .option("comment", "#")
    .csv(epss_csv_path)
)

epss_raw.printSchema()
epss_raw.show(5, truncate=False)

root
 |-- cve: string (nullable = true)
 |-- epss: string (nullable = true)
 |-- percentile: string (nullable = true)

+-------------+-------+----------+
|cve          |epss   |percentile|
+-------------+-------+----------+
|CVE-1999-0001|0.0119 |0.78876   |
|CVE-1999-0002|0.10103|0.93128   |
|CVE-1999-0003|0.90626|0.99621   |
|CVE-1999-0004|0.03372|0.87401   |
|CVE-1999-0005|0.1263 |0.94001   |
+-------------+-------+----------+
only showing top 5 rows


In [6]:
from pyspark.sql.functions import col, upper, trim

epss_df = (
    epss_raw
    .withColumnRenamed("cve", "cve_id")
    .withColumnRenamed("epss", "epss_score")
    .withColumnRenamed("percentile", "epss_percentile")
    .withColumn("cve_id", upper(trim(col("cve_id"))))
)

epss_df.show(5, truncate=False)

+-------------+----------+---------------+
|cve_id       |epss_score|epss_percentile|
+-------------+----------+---------------+
|CVE-1999-0001|0.0119    |0.78876        |
|CVE-1999-0002|0.10103   |0.93128        |
|CVE-1999-0003|0.90626   |0.99621        |
|CVE-1999-0004|0.03372   |0.87401        |
|CVE-1999-0005|0.1263    |0.94001        |
+-------------+----------+---------------+
only showing top 5 rows


In [7]:
from pyspark.sql.types import DoubleType

epss_df = (
    epss_df
    .withColumn("epss_score", col("epss_score").cast(DoubleType()))
    .withColumn("epss_percentile", col("epss_percentile").cast(DoubleType()))
)

In [8]:
print("Total rows:", epss_df.count())
print("Distinct CVEs:", epss_df.select("cve_id").distinct().count())

Total rows: 328654
Distinct CVEs: 328654


In [ ]:
epss_df.write.mode("overwrite").parquet("../data/silver/epss")

26/04/24 05:38:18 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 927363 ms exceeds timeout 120000 ms
26/04/24 05:38:18 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/24 05:38:25 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$